In [26]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n 2
!pip install -q mpi4py numba numpy


name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-5f3e9715-89c0-2858-ddd4-2c94ecde93ed, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [27]:
GITHUB_USERNAME = "AidynTt"

assert GITHUB_USERNAME.strip() != ""
print(f"Configured student GitHub identity: {GITHUB_USERNAME}")



Configured student GitHub identity: AidynTt


In [28]:

%%writefile distributed_cuda_solver.py
import os
import sys
import json
import math
import hashlib
import subprocess
import numpy as np
from mpi4py import MPI
from numba import cuda, float64


TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2
GLOBAL_NX = 512
GLOBAL_NY = 512
STEPS = 400
LAMBDA_DAMP = 0.15

def get_gpu_uuid() -> str:
    """Get the active NVIDIA GPU UUID."""
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8"
    )
    return out.strip().splitlines()[0].strip()

def derive_student_seed(github_user: str, gpu_uuid: str) -> int:
    """Generate a unique seed from GitHub username and GPU."""
    digest = hashlib.sha256(
        f"{github_user.strip()}::{gpu_uuid}".encode("utf-8")
    ).hexdigest()
    return int(digest[:16], 16)

def init_distributed_context(github_user: str):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()

    if size != 4:
        raise RuntimeError(
            f"Expected exactly 4 MPI ranks, got {size}."
        )

    cuda.select_device(0)

    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = comm.bcast(gpu_uuid, root=0)

    instance_seed = derive_student_seed(
        github_user, gpu_uuid
    )

    cart_comm = comm.Create_cart(
        dims=[2, 2],
        periods=[False, False],
        reorder=False
    )

    coords = cart_comm.Get_coords(rank)
    nbr_up, nbr_down = cart_comm.Shift(0, 1)
    nbr_left, nbr_right = cart_comm.Shift(1, 1)

    return (
        cart_comm, rank, coords,
        nbr_up, nbr_down,
        nbr_left, nbr_right,
        instance_seed
    )


Overwriting distributed_cuda_solver.py


In [29]:

%%writefile -a distributed_cuda_solver.py

# TASK 2: Initialize Subdomains
#I used ChatGPT

def initialize_subdomains(coords, instance_seed):
    ny, nx = 256, 256
    dx = 1.0 / 513.0
    dy = 1.0 / 513.0

    rows = coords[0] * ny + np.arange(258)
    cols = coords[1] * nx + np.arange(258)

    y = rows * dy
    x = cols * dx
    X, Y = np.meshgrid(x, y)

    kappa = 0.08 + 0.16 * (
        np.sin(np.pi * X) ** 2
        * np.sin(np.pi * Y) ** 2
    )

    rng = np.random.default_rng(instance_seed)
    cx, cy = rng.uniform(0.3, 0.7, size=2)

    u = np.exp(
        -((X - cx)**2 + (Y - cy)**2) / 0.01
    )

    source = np.exp(
        -((X - 0.5)**2 + (Y - 0.5)**2) / 0.002
    )

    return (
        cuda.to_device(u.astype(np.float64)),
        cuda.to_device(kappa.astype(np.float64)),
        cuda.to_device(source.astype(np.float64))
    )


Appending to distributed_cuda_solver.py


In [30]:

%%writefile -a distributed_cuda_solver.py

#TASK 3: GPU Halo Pack/Unpack

@cuda.jit
def pack_halos_kernel(u, top, bottom, left, right):
    idx = cuda.grid(1)
    ny = u.shape[0] - 2
    nx = u.shape[1] - 2

    if idx < nx:
        top[idx] = u[1, idx + 1]
        bottom[idx] = u[ny, idx + 1]

    if idx < ny:
        left[idx] = u[idx + 1, 1]
        right[idx] = u[idx + 1, nx]


@cuda.jit
def unpack_halos_kernel(u, top, bottom, left, right):
    idx = cuda.grid(1)
    ny = u.shape[0] - 2
    nx = u.shape[1] - 2

    if idx < nx:
        u[0, idx + 1] = top[idx]
        u[ny + 1, idx + 1] = bottom[idx]

    if idx < ny:
        u[idx + 1, 0] = left[idx]
        u[idx + 1, nx + 1] = right[idx]

Appending to distributed_cuda_solver.py


In [31]:

%%writefile -a distributed_cuda_solver.py

#TASK 3: Asynchronous MPI Halo Exchange

def exchange_halos_async(
    cart_comm, neighbors, send_host, recv_host
):
    opposite = (1, 0, 3, 2)
    requests = []

    for direction, neighbor in enumerate(neighbors):
        if neighbor == MPI.PROC_NULL:
            recv_host[direction].fill(0.0)
            continue

        requests.append(
            cart_comm.Irecv(
                recv_host[direction],
                source=neighbor,
                tag=opposite[direction]
            )
        )

    for direction, neighbor in enumerate(neighbors):
        if neighbor == MPI.PROC_NULL:
            continue

        requests.append(
            cart_comm.Isend(
                send_host[direction],
                dest=neighbor,
                tag=direction
            )
        )

    return requests


Appending to distributed_cuda_solver.py


In [32]:

%%writefile -a distributed_cuda_solver.py

#TASK 4: Shared-Memory Tiled Stencil
#I used ChatGpt

@cuda.jit
def tiled_stencil_kernel(
    u, u_next, kappa, source,
    dt, dx, dy,
    r_start, r_end, c_start, c_end
):
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    row = cuda.blockIdx.y * 16 + ty + r_start
    col = cuda.blockIdx.x * 16 + tx + c_start

    smem = cuda.shared.array(
        shape=(18, 18), dtype=float64
    )

    sy = ty + 1
    sx = tx + 1

    # Load center and halo values
    if row < u.shape[0] and col < u.shape[1]:
        smem[sy, sx] = u[row, col]

        if tx == 0 and col > 0:
            smem[sy, 0] = u[row, col - 1]

        if tx == 15 and col + 1 < u.shape[1]:
            smem[sy, 17] = u[row, col + 1]

        if ty == 0 and row > 0:
            smem[0, sx] = u[row - 1, col]

        if ty == 15 and row + 1 < u.shape[0]:
            smem[17, sx] = u[row + 1, col]

    cuda.syncthreads()

    if (
        row >= r_start and row <= r_end
        and col >= c_start and col <= c_end
    ):
        center = smem[sy, sx]

        up = smem[sy - 1, sx]
        down = smem[sy + 1, sx]
        left = smem[sy, sx - 1]
        right = smem[sy, sx + 1]

        diffusion = (
            (up + down - 2.0 * center) / (dy * dy)
            + (left + right - 2.0 * center) / (dx * dx)
        )

        u_next[row, col] = center + dt * (
            kappa[row, col] * diffusion
            - LAMBDA_DAMP * center ** 3
            + source[row, col]
        )


Appending to distributed_cuda_solver.py


In [33]:

from pathlib import Path

path = Path("distributed_cuda_solver.py")
code = path.read_text()

start = code.index("def initialize_subdomains(")
end = code.index("\n@cuda.jit", start)

replacement = '''
def initialize_subdomains(coords, instance_seed):
    ny, nx = 256, 256
    dx = dy = 1.0 / 513.0

    rows = coords[0] * ny + np.arange(258)
    cols = coords[1] * nx + np.arange(258)

    y = rows * dy
    x = cols * dx
    X, Y = np.meshgrid(x, y)

    kappa = 0.08 + 0.16 * (
        np.sin(np.pi * X)**2 *
        np.sin(np.pi * Y)**2
    )

    rng = np.random.default_rng(instance_seed)
    cx, cy = rng.uniform(0.3, 0.7, size=2)

    u = np.exp(
        -((X - cx)**2 + (Y - cy)**2) / 0.01
    )

    source = np.exp(
        -((X - 0.5)**2 + (Y - 0.5)**2) / 0.002
    )

    if coords[0] == 0:
        u[0, :] = 0.0
    if coords[0] == 1:
        u[-1, :] = 0.0
    if coords[1] == 0:
        u[:, 0] = 0.0
    if coords[1] == 1:
        u[:, -1] = 0.0

    return (
        cuda.to_device(u.astype(np.float64)),
        cuda.to_device(kappa.astype(np.float64)),
        cuda.to_device(source.astype(np.float64))
    )
'''

code = code[:start] + replacement + code[end:]
path.write_text(code)

print("Task 2 updated successfully!")


Task 2 updated successfully!


In [34]:
from pathlib import Path

path = Path("distributed_cuda_solver.py")
code = path.read_text()

code = code.replace(
    "rows = coords[0] * ny + np.arange(258)",
    "rows = coords[0] * ny + np.arange(258) - 1"
)

code = code.replace(
    "cols = coords[1] * nx + np.arange(258)",
    "cols = coords[1] * nx + np.arange(258) - 1"
)

path.write_text(code)

print("Global grid coordinates corrected!")


Global grid coordinates corrected!


In [35]:

import ast
from pathlib import Path

path = Path("distributed_cuda_solver.py")
code = path.read_text()

try:
    ast.parse(code)
    print("Python syntax: OK")
except SyntaxError as e:
    print(f"Syntax error at line {e.lineno}: {e.msg}")

print("\nCurrent functions:")
for line in code.splitlines():
    if line.startswith("def ") or line.startswith("@cuda.jit"):
        print(line)


Python syntax: OK

Current functions:
def get_gpu_uuid() -> str:
def derive_student_seed(github_user: str, gpu_uuid: str) -> int:
def init_distributed_context(github_user: str):
def initialize_subdomains(coords, instance_seed):
@cuda.jit
def pack_halos_kernel(u, top, bottom, left, right):
@cuda.jit
def unpack_halos_kernel(u, top, bottom, left, right):
def exchange_halos_async(
@cuda.jit
def tiled_stencil_kernel(


In [36]:

import ast
from pathlib import Path

path = Path("distributed_cuda_solver.py")
code = path.read_text()
lines = code.splitlines(keepends=True)
tree = ast.parse(code)

seen = set()
remove = []

for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        if node.name in seen:
            start = min(
                [node.lineno] +
                [d.lineno for d in node.decorator_list]
            )
            remove.append((start, node.end_lineno))
        else:
            seen.add(node.name)

for start, end in reversed(remove):
    del lines[start - 1:end]

path.write_text("".join(lines))

ast.parse(path.read_text())
print("Duplicate functions removed!")
print("Python syntax: OK")


Duplicate functions removed!
Python syntax: OK


In [37]:

from pathlib import Path
import shutil

source = Path("distributed_cuda_solver.py")
backup = Path("distributed_cuda_solver_backup.py")

shutil.copy2(source, backup)

print("Backup created successfully!")
print(f"Original file: {source}")
print(f"Backup file: {backup}")


Backup created successfully!
Original file: distributed_cuda_solver.py
Backup file: distributed_cuda_solver_backup.py


In [38]:

from pathlib import Path
import ast

path = Path("distributed_cuda_solver.py")
code = path.read_text()
tree = ast.parse(code)
lines = code.splitlines(keepends=True)

new_kernel = '''
@cuda.jit
def tiled_stencil_kernel(
    u, u_next, kappa, source,
    dt, dx, dy,
    r_start, r_end, c_start, c_end
):
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    row = cuda.blockIdx.y * 16 + ty + r_start
    col = cuda.blockIdx.x * 16 + tx + c_start

    sy = ty + 1
    sx = tx + 1

    smem = cuda.shared.array(
        shape=(18, 18), dtype=float64
    )

    # All threads initialize their shared-memory center
    valid = row < u.shape[0] and col < u.shape[1]

    if valid:
        smem[sy, sx] = u[row, col]
    else:
        smem[sy, sx] = 0.0

    if tx == 0:
        smem[sy, 0] = u[row, col - 1] if (
            row < u.shape[0] and 0 < col < u.shape[1]
        ) else 0.0

    if tx == 15:
        smem[sy, 17] = u[row, col + 1] if (
            row < u.shape[0] and col + 1 < u.shape[1]
        ) else 0.0

    if ty == 0:
        smem[0, sx] = u[row - 1, col] if (
            0 < row < u.shape[0] and col < u.shape[1]
        ) else 0.0

    if ty == 15:
        smem[17, sx] = u[row + 1, col] if (
            row + 1 < u.shape[0] and col < u.shape[1]
        ) else 0.0

    cuda.syncthreads()

    if (
        r_start <= row <= r_end
        and c_start <= col <= c_end
    ):
        center = smem[sy, sx]

        up = smem[sy - 1, sx]
        down = smem[sy + 1, sx]
        left = smem[sy, sx - 1]
        right = smem[sy, sx + 1]

        kc = kappa[row, col]
        ku = 0.5 * (kc + kappa[row - 1, col])
        kd = 0.5 * (kc + kappa[row + 1, col])
        kl = 0.5 * (kc + kappa[row, col - 1])
        kr = 0.5 * (kc + kappa[row, col + 1])

        diffusion = (
            (kd * (down - center)
             - ku * (center - up)) / (dy * dy)
            +
            (kr * (right - center)
             - kl * (center - left)) / (dx * dx)
        )

        u_next[row, col] = center + dt * (
            diffusion
            - 0.15 * center ** 3
            + source[row, col]
        )
'''

target = next(
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "tiled_stencil_kernel"
)

start = min(
    [target.lineno] +
    [d.lineno for d in target.decorator_list]
)
end = target.end_lineno

lines[start - 1:end] = [new_kernel + "\n"]

updated = "".join(lines)
ast.parse(updated)
path.write_text(updated)

print("CUDA stencil kernel updated!")
print("Python syntax: OK")


CUDA stencil kernel updated!
Python syntax: OK


In [39]:

import numpy as np
from numba import cuda
import importlib.util

spec = importlib.util.spec_from_file_location(
    "solver", "distributed_cuda_solver.py"
)
solver = importlib.util.module_from_spec(spec)
spec.loader.exec_module(solver)

u = np.ones((258, 258), dtype=np.float64)
kappa = np.full((258, 258), 0.1, dtype=np.float64)
source = np.zeros((258, 258), dtype=np.float64)

d_u = cuda.to_device(u)
d_next = cuda.device_array_like(d_u)
d_kappa = cuda.to_device(kappa)
d_source = cuda.to_device(source)

dt = 1e-6
dx = dy = 1.0 / 513.0

solver.tiled_stencil_kernel[
    (16, 16), (16, 16)
](
    d_u, d_next, d_kappa, d_source,
    dt, dx, dy,
    1, 256, 1, 256
)

cuda.synchronize()

result = d_next.copy_to_host()
expected = 1.0 - dt * 0.15

error = np.max(
    np.abs(result[1:257, 1:257] - expected)
)

print("CUDA kernel executed successfully!")
print("Maximum error:", error)
print("Test passed:", error < 1e-12)


CUDA kernel executed successfully!
Maximum error: 0.0
Test passed: True


In [40]:

import numpy as np
from numba import cuda

u = np.ones((258, 258), dtype=np.float64)
kappa = np.full((258, 258), 0.1, dtype=np.float64)
source = np.zeros((258, 258), dtype=np.float64)

d_u = cuda.to_device(u)
d_next = cuda.to_device(np.zeros_like(u))
d_kappa = cuda.to_device(kappa)
d_source = cuda.to_device(source)

# Test a region that is not divisible by 16
r_start, r_end = 2, 254
c_start, c_end = 2, 254

rows = r_end - r_start + 1
cols = c_end - c_start + 1

blocks = (
    (cols + 15) // 16,
    (rows + 15) // 16
)

solver.tiled_stencil_kernel[
    blocks, (16, 16)
](
    d_u, d_next, d_kappa, d_source,
    1e-6, 1.0 / 513, 1.0 / 513,
    r_start, r_end, c_start, c_end
)

cuda.synchronize()

result = d_next.copy_to_host()
expected = 1.0 - 1e-6 * 0.15

error = np.max(np.abs(
    result[r_start:r_end+1, c_start:c_end+1]
    - expected
))

print("Partial-block CUDA test completed!")
print("Maximum error:", error)
print("Test passed:", error < 1e-12)


Partial-block CUDA test completed!
Maximum error: 0.0
Test passed: True


In [41]:

import numpy as np
from numba import cuda

rng = np.random.default_rng(42)

u = rng.random((258, 258))
kappa = np.full((258, 258), 0.1)
source = np.zeros((258, 258))

dt = 1e-7
dx = dy = 1.0 / 513.0

# Independent CPU reference calculation
expected = u.copy()

for i in range(1, 257):
    for j in range(1, 257):
        laplacian = (
            (u[i+1, j] - 2*u[i, j] + u[i-1, j]) / dy**2
            + (u[i, j+1] - 2*u[i, j] + u[i, j-1]) / dx**2
        )

        expected[i, j] = u[i, j] + dt * (
            0.1 * laplacian - 0.15 * u[i, j]**3
        )

d_u = cuda.to_device(u)
d_next = cuda.to_device(np.zeros_like(u))
d_kappa = cuda.to_device(kappa)
d_source = cuda.to_device(source)

solver.tiled_stencil_kernel[
    (16, 16), (16, 16)
](
    d_u, d_next, d_kappa, d_source,
    dt, dx, dy,
    1, 256, 1, 256
)

cuda.synchronize()
actual = d_next.copy_to_host()

error = np.max(np.abs(
    actual[1:257, 1:257] -
    expected[1:257, 1:257]
))

print("Non-uniform CUDA test completed!")
print("Maximum error:", error)
print("Test passed:", error < 1e-12)


Non-uniform CUDA test completed!
Maximum error: 1.1102230246251565e-16
Test passed: True


In [42]:

%%writefile test_mpi_context.py
import sys
from mpi4py import MPI
from distributed_cuda_solver import init_distributed_context

github_user = sys.argv[1]

(
    cart_comm, rank, coords,
    nbr_up, nbr_down,
    nbr_left, nbr_right,
    instance_seed
) = init_distributed_context(github_user)

print(
    f"Rank {rank}: coords={coords}, "
    f"neighbors=({nbr_up}, {nbr_down}, "
    f"{nbr_left}, {nbr_right}), "
    f"seed={instance_seed}",
    flush=True
)

cart_comm.Barrier()


Overwriting test_mpi_context.py


In [43]:

import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "test_mpi_context.py",
    GITHUB_USERNAME
]

proc = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    timeout=120
)

print(proc.stdout)

if proc.returncode != 0:
    print("ERROR:", proc.stderr)
else:
    print("MPI topology test passed!")



ERROR: 


In [44]:

print("Return code:", proc.returncode)
print("STDOUT:", repr(proc.stdout))
print("STDERR:", repr(proc.stderr))


Return code: 1
STDOUT: ''
STDERR: ''


In [45]:

import subprocess

tests = [
    ["mpirun", "--allow-run-as-root",
     "--oversubscribe", "-np", "4",
     "hostname"],

    ["python3", "test_mpi_context.py",
     GITHUB_USERNAME]
]

for i, cmd in enumerate(tests, 1):
    print(f"\nTEST {i}")
    try:
        p = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=30
        )
        print("Return code:", p.returncode)
        print("STDOUT:", p.stdout[:2000])
        print("STDERR:", p.stderr[:2000])
    except Exception as e:
        print("Exception:", repr(e))



TEST 1
Return code: 1
STDOUT: 
STDERR: 

TEST 2
Return code: 1
STDOUT: 
STDERR: Traceback (most recent call last):
  File "/content/test_mpi_context.py", line 12, in <module>
    ) = init_distributed_context(github_user)
        ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "/content/distributed_cuda_solver.py", line 40, in init_distributed_context
    raise RuntimeError(
        f"Expected exactly 4 MPI ranks, got {size}."
    )
RuntimeError: Expected exactly 4 MPI ranks, got 1.



In [46]:

import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--mca", "plm", "isolated",
    "--mca", "btl", "self,tcp",
    "-np", "4",
    "hostname"
]

p = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    timeout=30
)

print("Return code:", p.returncode)
print("STDOUT:", repr(p.stdout))
print("STDERR:", repr(p.stderr))


Return code: 1
STDOUT: ''
STDERR: ''


In [47]:

import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--mca", "plm_base_verbose", "10",
    "-np", "4",
    "/bin/hostname"
]

p = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    timeout=30
)

print("Return code:", p.returncode)
print("Output:")
print(repr(p.stdout[:6000]))


Return code: 1
Output:
''


In [48]:

import subprocess
import shutil
import os

print("mpirun path:", shutil.which("mpirun"))
print("User ID:", os.getuid())

commands = [
    ["mpirun", "--version"],
    ["mpirun", "--allow-run-as-root", "-np", "1", "/bin/echo", "HELLO"],
    ["mpirun", "--allow-run-as-root", "-np", "4", "/bin/echo", "HELLO"]
]

for cmd in commands:
    print("\nCOMMAND:", " ".join(cmd))
    try:
        p = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=15
        )
        print("Return code:", p.returncode)
        print("STDOUT:", repr(p.stdout[:1000]))
        print("STDERR:", repr(p.stderr[:1000]))
    except Exception as e:
        print("ERROR:", repr(e))


mpirun path: /usr/bin/mpirun
User ID: 0

COMMAND: mpirun --version
Return code: 0
STDOUT: 'mpirun (Open MPI) 4.1.6\n\nReport bugs to http://www.open-mpi.org/community/help/\n'
STDERR: ''

COMMAND: mpirun --allow-run-as-root -np 1 /bin/echo HELLO
Return code: 1
STDOUT: ''
STDERR: ''

COMMAND: mpirun --allow-run-as-root -np 4 /bin/echo HELLO
Return code: 1
STDOUT: ''
STDERR: ''


In [49]:

import subprocess

commands = [
    "which mpirun; which prterun; which orted",
    "ldd /usr/bin/mpirun | grep 'not found' || true",
    "ompi_info --parsable | head -n 8",
    "ls -ld /tmp /dev/shm",
    "df -h /tmp /dev/shm",
    "env | grep -E '^(OMPI|PMIX|PMI|PRTE)_' || true"
]

for cmd in commands:
    print("\nCOMMAND:", cmd)
    p = subprocess.run(
        ["bash", "-lc", cmd],
        capture_output=True,
        text=True,
        timeout=20
    )
    print("Return code:", p.returncode)
    print("STDOUT:", p.stdout[:1500])
    print("STDERR:", p.stderr[:1500])



COMMAND: which mpirun; which prterun; which orted
Return code: 0
STDOUT: /usr/bin/mpirun
/usr/bin/orted

STDERR: 

COMMAND: ldd /usr/bin/mpirun | grep 'not found' || true
Return code: 0
STDOUT: 
STDERR: 

COMMAND: ompi_info --parsable | head -n 8
Return code: 0
STDOUT: package:Debian OpenMPI
ompi:version:full:4.1.6
ompi:version:repo:v4.1.6
ompi:version:release_date:Sep 30, 2023
orte:version:full:4.1.6
orte:version:repo:v4.1.6
orte:version:release_date:Sep 30, 2023
opal:version:full:4.1.6

STDERR: 

COMMAND: ls -ld /tmp /dev/shm
Return code: 0
STDOUT: drwxrwxrwt 2 root root   40 Nov 20 05:36 /dev/shm
drwxrwxrwt 1 root root 4096 Nov 20 05:35 /tmp

STDERR: 

COMMAND: df -h /tmp /dev/shm
Return code: 0
STDOUT: Filesystem      Size  Used Avail Use% Mounted on
overlay         113G  101M  113G   1% /
shm             5.7G     0  5.7G   0% /dev/shm

STDERR: 

COMMAND: env | grep -E '^(OMPI|PMIX|PMI|PRTE)_' || true
Return code: 0
STDOUT: OMPI_MCA_orte_precondition_transports=d14dedec88585cdc-03

In [50]:

import os
import subprocess

# Remove inherited MPI runtime variables
clean_env = {
    key: value
    for key, value in os.environ.items()
    if not key.startswith((
        "OMPI_", "PMIX_", "PMI_", "PRTE_"
    ))
}

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "/bin/echo", "HELLO"
]

p = subprocess.run(
    cmd,
    env=clean_env,
    capture_output=True,
    text=True,
    timeout=30
)

print("Return code:", p.returncode)
print("STDOUT:", p.stdout)
print("STDERR:", p.stderr)


Return code: 0
STDOUT: HELLO
HELLO
HELLO
HELLO

STDERR: 


In [51]:

import os
import subprocess

clean_env = {
    k: v for k, v in os.environ.items()
    if not k.startswith(("OMPI_", "PMIX_", "PMI_", "PRTE_"))
}

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "test_mpi_context.py",
    GITHUB_USERNAME
]

p = subprocess.run(
    cmd,
    env=clean_env,
    capture_output=True,
    text=True,
    timeout=120
)

print("Return code:", p.returncode)
print("STDOUT:\n", p.stdout)
print("STDERR:\n", p.stderr)


Return code: 0
STDOUT:
 Rank 0: coords=[0, 0], neighbors=(-2, 2, -2, 1), seed=15483842694073859920
Rank 1: coords=[0, 1], neighbors=(-2, 3, 0, -2), seed=15483842694073859920
Rank 2: coords=[1, 0], neighbors=(0, -2, -2, 3), seed=15483842694073859920
Rank 3: coords=[1, 1], neighbors=(1, -2, 2, -2), seed=15483842694073859920

STDERR:
 


In [52]:

%%writefile test_halo_exchange.py
import sys
import numpy as np
from mpi4py import MPI
from distributed_cuda_solver import init_distributed_context, exchange_halos_async

(
    cart_comm, rank, coords,
    up, down, left, right,
    seed
) = init_distributed_context(sys.argv[1])

neighbors = (up, down, left, right)

# Each rank sends its rank number
send_host = [
    np.full(256, rank, dtype=np.float64)
    for _ in range(4)
]

recv_host = [
    np.zeros(256, dtype=np.float64)
    for _ in range(4)
]

requests = exchange_halos_async(
    cart_comm, neighbors, send_host, recv_host
)

MPI.Request.Waitall(requests)

passed = True

for direction, neighbor in enumerate(neighbors):
    expected = 0 if neighbor == MPI.PROC_NULL else neighbor

    if not np.all(recv_host[direction] == expected):
        passed = False

all_passed = cart_comm.allreduce(
    passed, op=MPI.LAND
)

print(
    f"Rank {rank}: halo exchange "
    f"{'PASSED' if passed else 'FAILED'}",
    flush=True
)

if rank == 0:
    print("All ranks passed:", all_passed, flush=True)

cart_comm.Barrier()


Writing test_halo_exchange.py


In [53]:

import subprocess

p = subprocess.run(
    [
        "mpirun",
        "--allow-run-as-root",
        "--oversubscribe",
        "-np", "4",
        "python3",
        "test_halo_exchange.py",
        GITHUB_USERNAME
    ],
    env=clean_env,
    capture_output=True,
    text=True,
    timeout=120
)

print("Return code:", p.returncode)
print("STDOUT:\n", p.stdout)
print("STDERR:\n", p.stderr)


Return code: 0
STDOUT:
 Rank 0: halo exchange PASSED
All ranks passed: True
Rank 2: halo exchange PASSED
Rank 3: halo exchange PASSED
Rank 1: halo exchange PASSED

STDERR:
 


In [54]:

import numpy as np
from numba import cuda

#AI assistance: ChatGPT

stream_boundary = cuda.stream()
stream_interior = cuda.stream()

directions = ["top", "bottom", "left", "right"]

send_host = {
    d: cuda.pinned_array(256, dtype=np.float64)
    for d in directions
}

recv_host = {
    d: cuda.pinned_array(256, dtype=np.float64)
    for d in directions
}

send_device = {
    d: cuda.device_array(256, dtype=np.float64)
    for d in directions
}

recv_device = {
    d: cuda.device_array(256, dtype=np.float64)
    for d in directions
}

print("CUDA streams created: 2")
print("Pinned host buffers created:", len(send_host) + len(recv_host))
print("GPU buffers created:", len(send_device) + len(recv_device))
print("GPU:", cuda.get_current_device().name)


CUDA streams created: 2
Pinned host buffers created: 8
GPU buffers created: 8
GPU: Tesla T4


In [55]:

import numpy as np
from numba import cuda

#AI assistance: ChatGPT

test_data = np.arange(256, dtype=np.float64)

d_source = cuda.to_device(test_data)
d_target = cuda.device_array(256, dtype=np.float64)

host_send = cuda.pinned_array(256, dtype=np.float64)
host_recv = cuda.pinned_array(256, dtype=np.float64)

# GPU to pinned CPU memory
d_source.copy_to_host(
    host_send,
    stream=stream_boundary
)
stream_boundary.synchronize()

# Simulate received MPI data
np.copyto(host_recv, host_send)

# Pinned CPU memory to GPU
d_target.copy_to_device(
    host_recv,
    stream=stream_boundary
)
stream_boundary.synchronize()

result = d_target.copy_to_host()

error = np.max(np.abs(result - test_data))

print("GPU -> CPU -> GPU transfer completed!")
print("Maximum error:", error)
print("Test passed:", error == 0.0)


GPU -> CPU -> GPU transfer completed!
Maximum error: 0.0
Test passed: True


In [56]:

import numpy as np
from numba import cuda

u = np.zeros((258, 258), dtype=np.float64)

# Fill interior cells with test values
for i in range(1, 257):
    for j in range(1, 257):
        u[i, j] = i * 1000 + j

d_u = cuda.to_device(u)

d_top = cuda.device_array(256, dtype=np.float64)
d_bottom = cuda.device_array(256, dtype=np.float64)
d_left = cuda.device_array(256, dtype=np.float64)
d_right = cuda.device_array(256, dtype=np.float64)

solver.pack_halos_kernel[1, 256](
    d_u, d_top, d_bottom, d_left, d_right
)

cuda.synchronize()

top = d_top.copy_to_host()
bottom = d_bottom.copy_to_host()
left = d_left.copy_to_host()
right = d_right.copy_to_host()

passed = (
    np.array_equal(top, u[1, 1:257])
    and np.array_equal(bottom, u[256, 1:257])
    and np.array_equal(left, u[1:257, 1])
    and np.array_equal(right, u[1:257, 256])
)

print("GPU Halo Packing completed!")
print("Top:", top[:5])
print("Left:", left[:5])
print("Test passed:", passed)


GPU Halo Packing completed!
Top: [1001. 1002. 1003. 1004. 1005.]
Left: [1001. 2001. 3001. 4001. 5001.]
Test passed: True


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [57]:

import numpy as np
from numba import cuda

# AI assistance: ChatGPT (OpenAI)

u = np.zeros((258, 258), dtype=np.float64)
d_u = cuda.to_device(u)

d_top = cuda.to_device(np.full(256, 10.0))
d_bottom = cuda.to_device(np.full(256, 20.0))
d_left = cuda.to_device(np.full(256, 30.0))
d_right = cuda.to_device(np.full(256, 40.0))

solver.unpack_halos_kernel[1, 256](
    d_u, d_top, d_bottom, d_left, d_right
)

cuda.synchronize()
result = d_u.copy_to_host()

passed = (
    np.all(result[0, 1:257] == 10.0)
    and np.all(result[257, 1:257] == 20.0)
    and np.all(result[1:257, 0] == 30.0)
    and np.all(result[1:257, 257] == 40.0)
)

print("GPU Halo Unpacking completed!")
print("Top ghost cells:", result[0, 1:6])
print("Left ghost cells:", result[1:6, 0])
print("Test passed:", passed)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


GPU Halo Unpacking completed!
Top ghost cells: [10. 10. 10. 10. 10.]
Left ghost cells: [30. 30. 30. 30. 30.]
Test passed: True


In [58]:

%%writefile test_gpu_mpi_halo.py
import sys
import numpy as np
from mpi4py import MPI
from numba import cuda
from distributed_cuda_solver import (
    init_distributed_context,
    pack_halos_kernel,
    unpack_halos_kernel,
    exchange_halos_async
)

(
    cart, rank, coords,
    up, down, left, right, seed
) = init_distributed_context(sys.argv[1])

neighbors = (up, down, left, right)
stream = cuda.stream()

u = np.zeros((258, 258), dtype=np.float64)
u[1:257, 1:257] = rank + 1
d_u = cuda.to_device(u)

send_device = [
    cuda.device_array(256, dtype=np.float64)
    for _ in range(4)
]
recv_device = [
    cuda.device_array(256, dtype=np.float64)
    for _ in range(4)
]

send_host = [
    cuda.pinned_array(256, dtype=np.float64)
    for _ in range(4)
]
recv_host = [
    cuda.pinned_array(256, dtype=np.float64)
    for _ in range(4)
]

pack_halos_kernel[1, 256, stream](
    d_u, *send_device
)

for i in range(4):
    send_device[i].copy_to_host(
        send_host[i], stream=stream
    )

stream.synchronize()

requests = exchange_halos_async(
    cart, neighbors, send_host, recv_host
)
MPI.Request.Waitall(requests)

for i in range(4):
    recv_device[i].copy_to_device(
        recv_host[i], stream=stream
    )

unpack_halos_kernel[1, 256, stream](
    d_u, *recv_device
)
stream.synchronize()

result = d_u.copy_to_host()

edges = [
    result[0, 1:257],
    result[257, 1:257],
    result[1:257, 0],
    result[1:257, 257]
]

passed = all(
    np.all(edge == (
        0 if neighbor == MPI.PROC_NULL
        else neighbor + 1
    ))
    for edge, neighbor in zip(edges, neighbors)
)

all_passed = cart.allreduce(
    passed, op=MPI.LAND
)

print(f"Rank {rank}: GPU-MPI halo "
      f"{'PASSED' if passed else 'FAILED'}",
      flush=True)

if rank == 0:
    print("All ranks passed:", all_passed,
          flush=True)

cart.Barrier()


Writing test_gpu_mpi_halo.py


In [59]:

import subprocess

p = subprocess.run(
    [
        "mpirun", "--allow-run-as-root",
        "--oversubscribe", "-np", "4",
        "python3", "test_gpu_mpi_halo.py",
        GITHUB_USERNAME
    ],
    env=clean_env,
    capture_output=True,
    text=True,
    timeout=120
)

print("Return code:", p.returncode)
print("STDOUT:\n", p.stdout)
print("STDERR:\n", p.stderr)


Return code: 0
STDOUT:
 Rank 0: GPU-MPI halo PASSED
All ranks passed: True
Rank 1: GPU-MPI halo PASSED
Rank 2: GPU-MPI halo PASSED
Rank 3: GPU-MPI halo PASSED

STDERR:
 /usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU und

In [60]:

import ast

with open("distributed_cuda_solver.py", "r") as f:
    source = f.read()

tree = ast.parse(source)

print("=== FUNCTIONS ===")
for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        print(f"{node.name} (line {node.lineno})")

print("\n=== FILE SIZE ===")
print(len(source.splitlines()), "lines")


=== FUNCTIONS ===
get_gpu_uuid (line 19)
derive_student_seed (line 27)
init_distributed_context (line 34)
initialize_subdomains (line 74)
pack_halos_kernel (line 117)
unpack_halos_kernel (line 132)
exchange_halos_async (line 147)
tiled_stencil_kernel (line 185)

=== FILE SIZE ===
263 lines


In [61]:

import inspect
import distributed_cuda_solver as solver

for name in [
    "initialize_subdomains",
    "exchange_halos_async",
    "tiled_stencil_kernel"
]:
    fn = getattr(solver, name)
    print(f"\n=== {name} ===")
    print(inspect.getsource(fn))



=== initialize_subdomains ===
def initialize_subdomains(coords, instance_seed):
    ny, nx = 256, 256
    dx = dy = 1.0 / 513.0

    rows = coords[0] * ny + np.arange(258) - 1
    cols = coords[1] * nx + np.arange(258) - 1

    y = rows * dy
    x = cols * dx
    X, Y = np.meshgrid(x, y)

    kappa = 0.08 + 0.16 * (
        np.sin(np.pi * X)**2 *
        np.sin(np.pi * Y)**2
    )

    rng = np.random.default_rng(instance_seed)
    cx, cy = rng.uniform(0.3, 0.7, size=2)

    u = np.exp(
        -((X - cx)**2 + (Y - cy)**2) / 0.01
    )

    source = np.exp(
        -((X - 0.5)**2 + (Y - 0.5)**2) / 0.002
    )

    if coords[0] == 0:
        u[0, :] = 0.0
    if coords[0] == 1:
        u[-1, :] = 0.0
    if coords[1] == 0:
        u[:, 0] = 0.0
    if coords[1] == 1:
        u[:, -1] = 0.0

    return (
        cuda.to_device(u.astype(np.float64)),
        cuda.to_device(kappa.astype(np.float64)),
        cuda.to_device(source.astype(np.float64))
    )


=== exchange_halos_async ===
de

In [62]:

%%writefile -a distributed_cuda_solver.py

# ============================================================
# Full 400-step solver
# AI assistance: ChatGPT (OpenAI)
# ============================================================

import time
import re

DIRECTIONS = ("top", "bottom", "left", "right")


def launch_stencil(u, out, kappa, source,
                   dt, dx, dy, box, stream):
    r0, r1, c0, c1 = box
    width = c1 - c0 + 1
    height = r1 - r0 + 1

    blocks = (
        (width + 15) // 16,
        (height + 15) // 16
    )

    tiled_stencil_kernel[blocks, (16, 16), stream](
        u, out, kappa, source,
        dt, dx, dy,
        r0, r1, c0, c1
    )


def step_overlapped(
    cart, neighbors, u, out, kappa, source,
    dt, dx, dy, buffers, stream_boundary,
    stream_interior, overlap=True
):
    sd, rd, sh, rh = buffers

    # Pack four GPU boundaries
    pack_halos_kernel[1, 256, stream_boundary](
        u, *sd
    )

    # GPU -> pinned host
    for i in range(4):
        sd[i].copy_to_host(
            sh[i], stream=stream_boundary
        )

    stream_boundary.synchronize()

    # Start non-blocking MPI communication
    requests = exchange_halos_async(
        cart, neighbors, sh, rh
    )

    # Deep interior is independent of incoming halos
    if overlap:
        launch_stencil(
            u, out, kappa, source,
            dt, dx, dy,
            (2, 255, 2, 255),
            stream_interior
        )

    MPI.Request.Waitall(requests)

    # Pinned host -> GPU
    for i in range(4):
        rd[i].copy_to_device(
            rh[i], stream=stream_boundary
        )

    unpack_halos_kernel[1, 256, stream_boundary](
        u, *rd
    )

    # In serialized mode, wait for communication first
    if not overlap:
        stream_boundary.synchronize()
        launch_stencil(
            u, out, kappa, source,
            dt, dx, dy,
            (2, 255, 2, 255),
            stream_interior
        )

    # Four non-overlapping perimeter regions
    boundary_boxes = [
        (1, 1, 1, 256),
        (256, 256, 1, 256),
        (2, 255, 1, 1),
        (2, 255, 256, 256)
    ]

    for box in boundary_boxes:
        launch_stencil(
            u, out, kappa, source,
            dt, dx, dy,
            box, stream_boundary
        )

    stream_boundary.synchronize()
    stream_interior.synchronize()

    return out, u


def distributed_metrics(cart, u, previous, dx, dy):
    current = u.copy_to_host()
    old = previous.copy_to_host()

    inside = current[1:257, 1:257]
    difference = (
        inside - old[1:257, 1:257]
    )

    local_energy = float(
        np.sum(inside) * dx * dy
    )
    local_residual_sq = float(
        np.sum(difference ** 2) * dx * dy
    )

    local_values = np.array(
        [local_energy, local_residual_sq],
        dtype=np.float64
    )
    global_values = np.zeros_like(local_values)

    cart.Allreduce(
        local_values,
        global_values,
        op=MPI.SUM
    )

    return (
        float(global_values[0]),
        float(np.sqrt(global_values[1]))
    )


def run_mode(cart, coords, neighbors, seed, mode):
    dx = dy = 1.0 / 513.0

    # Stable explicit time step
    dt = 0.1 * min(dx, dy) ** 2 / 0.24

    u, kappa, source = initialize_subdomains(
        coords, seed
    )
    out = cuda.device_array_like(u)

    sb = cuda.stream()
    si = cuda.stream()

    sd = [
        cuda.device_array(256, dtype=np.float64)
        for _ in range(4)
    ]
    rd = [
        cuda.device_array(256, dtype=np.float64)
        for _ in range(4)
    ]
    sh = [
        cuda.pinned_array(256, dtype=np.float64)
        for _ in range(4)
    ]
    rh = [
        cuda.pinned_array(256, dtype=np.float64)
        for _ in range(4)
    ]

    buffers = (sd, rd, sh, rh)

    rank = cart.Get_rank()
    trace = []

    energy0, _ = distributed_metrics(
        cart, u, u, dx, dy
    )
    trace.append({
        "step": 0,
        "energy": energy0,
        "residual": 0.0
    })

    if rank == 0:
        print(
            f"{mode} step 0: "
            f"energy={energy0:.12e}",
            flush=True
        )

    cuda.synchronize()
    cart.Barrier()
    start = time.perf_counter()

    for step in range(1, STEPS + 1):
        u, out = step_overlapped(
            cart, neighbors, u, out,
            kappa, source, dt, dx, dy,
            buffers, sb, si,
            overlap=(mode == "overlapped")
        )

        if step % 100 == 0:
            energy, residual = distributed_metrics(
                cart, u, out, dx, dy
            )
            trace.append({
                "step": step,
                "energy": energy,
                "residual": residual
            })

            if rank == 0:
                print(
                    f"{mode} step {step}: "
                    f"energy={energy:.12e}, "
                    f"residual={residual:.12e}",
                    flush=True
                )

    cuda.synchronize()
    cart.Barrier()

    local_ms = (time.perf_counter() - start) * 1000
    elapsed_ms = cart.allreduce(
        local_ms, op=MPI.MAX
    )

    if rank == 0:
        print(
            f"{mode} time: {elapsed_ms:.3f} ms",
            flush=True
        )

    return u.copy_to_host(), elapsed_ms, trace


def audit_ptx():
    asm = ""

    for signature in tiled_stencil_kernel.signatures:
        try:
            asm = tiled_stencil_kernel.inspect_asm(
                signature
            )
            if asm:
                break
        except Exception:
            pass

    counts = {
        instruction: len(
            re.findall(
                r"\b" + re.escape(instruction) + r"\b",
                asm
            )
        )
        for instruction in (
            "st.shared", "ld.shared", "bar.sync"
        )
    }

    return counts, all(v > 0 for v in counts.values())


def run_solver(github_user):
    (
        cart, rank, coords,
        up, down, left, right, seed
    ) = init_distributed_context(github_user)

    neighbors = (up, down, left, right)

    if rank == 0:
        print("MPI ranks: 4")
        print("Cartesian grid: 2 x 2")
        print("Time steps:", STEPS)
        print("Instance seed:", seed, flush=True)

    serial_u, serial_ms, serial_trace = run_mode(
        cart, coords, neighbors,
        seed, "serialized"
    )

    overlap_u, overlap_ms, overlap_trace = run_mode(
        cart, coords, neighbors,
        seed, "overlapped"
    )

    local_diff = float(np.max(
        np.abs(serial_u - overlap_u)
    ))

    parity = cart.allreduce(
        local_diff, op=MPI.MAX
    )

    ptx_counts, ptx_passed = audit_ptx()

    all_ptx_passed = cart.allreduce(
        ptx_passed, op=MPI.LAND
    )

    if rank == 0:
        speedup = (
            serial_ms / overlap_ms
            if overlap_ms > 0 else None
        )

        manifest = {
            "github_username": github_user,
            "gpu_uuid": get_gpu_uuid(),
            "instance_seed": seed,
            "mpi_ranks": 4,
            "cartesian_grid": [2, 2],
            "global_grid": [512, 512],
            "time_steps": STEPS,
            "serialized_ms": serial_ms,
            "overlapped_ms": overlap_ms,
            "speedup": speedup,
            "parity_max_diff": parity,
            "parity_passed": parity < 1e-12,
            "ptx_counts_rank0": ptx_counts,
            "ptx_audit_passed": bool(all_ptx_passed),
            "serialized_trace": serial_trace,
            "overlapped_trace": overlap_trace
        }

        with open(
            "submission_manifest.json", "w"
        ) as f:
            json.dump(manifest, f, indent=2)

        print("\n=== FINAL RESULTS ===")
        print("Serialized:", serial_ms, "ms")
        print("Overlapped:", overlap_ms, "ms")
        print("Speedup:", speedup)
        print("Parity max diff:", parity)
        print("Parity passed:", parity < 1e-12)
        print("PTX counts:", ptx_counts)
        print("PTX audit passed:", all_ptx_passed)
        print(
            "Manifest: submission_manifest.json",
            flush=True
        )

    cart.Barrier()


if __name__ == "__main__":
    if len(sys.argv) < 2:
        raise SystemExit(
            "Usage: python distributed_cuda_solver.py "
            "<github_username>"
        )

    run_solver(sys.argv[1])


Appending to distributed_cuda_solver.py


In [63]:

import os
import subprocess

clean_env = {
    k: v for k, v in os.environ.items()
    if not k.startswith(
        ("OMPI_", "PMIX_", "PMI_", "PRTE_")
    )
}

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME
]

try:
    p = subprocess.run(
        cmd,
        env=clean_env,
        capture_output=True,
        text=True,
        timeout=900
    )

    print("Return code:", p.returncode)
    print("STDOUT:\n", p.stdout)
    print("STDERR:\n", p.stderr)

except subprocess.TimeoutExpired as e:
    print("Execution exceeded 15 minutes.")
    print("Partial STDOUT:", e.stdout)
    print("Partial STDERR:", e.stderr)


Return code: 0
STDOUT:
 MPI ranks: 4
Cartesian grid: 2 x 2
Time steps: 400
Instance seed: 15483842694073859920
serialized step 0: energy=3.141592560659e-02
serialized step 100: energy=3.141667448758e-02, residual=1.120971385662e-05
serialized step 200: energy=3.141742952836e-02, residual=1.098618197357e-05
serialized step 300: energy=3.141819042250e-02, residual=1.077027204580e-05
serialized step 400: energy=3.141895693783e-02, residual=1.056161182606e-05
serialized time: 8065.382 ms
overlapped step 0: energy=3.141592560659e-02
overlapped step 100: energy=3.141667448758e-02, residual=1.120971385662e-05
overlapped step 200: energy=3.141742952836e-02, residual=1.098618197357e-05
overlapped step 300: energy=3.141819042250e-02, residual=1.077027204580e-05
overlapped step 400: energy=3.141895693783e-02, residual=1.056161182606e-05
overlapped time: 2235.619 ms

=== FINAL RESULTS ===
Serialized: 8065.382182000121 ms
Overlapped: 2235.6187339996723 ms
Speedup: 3.6076733744177436
Parity max diff

In [64]:

import numpy as np
from numba import cuda
import distributed_cuda_solver as solver

# AI assistance: ChatGPT
rng = np.random.default_rng(12345)

u = rng.random((258, 258))
kappa = 0.08 + 0.16 * rng.random((258, 258))
source = rng.random((258, 258)) * 0.01

dx = dy = 1.0 / 513.0
dt = 0.1 * dx**2 / 0.24

d_u = cuda.to_device(u)
d_k = cuda.to_device(kappa)
d_s = cuda.to_device(source)
d_out = cuda.to_device(np.zeros_like(u))

solver.tiled_stencil_kernel[(16, 16), (16, 16)](
    d_u, d_out, d_k, d_s,
    dt, dx, dy, 1, 256, 1, 256
)
cuda.synchronize()

gpu = d_out.copy_to_host()[1:257, 1:257]

c = u[1:257, 1:257]
up = u[0:256, 1:257]
down = u[2:258, 1:257]
left = u[1:257, 0:256]
right = u[1:257, 2:258]

kc = kappa[1:257, 1:257]
ku = (kc + kappa[0:256, 1:257]) / 2
kd = (kc + kappa[2:258, 1:257]) / 2
kl = (kc + kappa[1:257, 0:256]) / 2
kr = (kc + kappa[1:257, 2:258]) / 2

reference = c + dt * (
    (kd * (down-c) - ku * (c-up)) / dy**2
    + (kr * (right-c) - kl * (c-left)) / dx**2
    - 0.15 * c**3
    + source[1:257, 1:257]
)

error = np.max(np.abs(gpu - reference))
print("Maximum GPU vs NumPy error:", error)
print("Full-grid stencil passed:", error < 1e-12)


Maximum GPU vs NumPy error: 1.1102230246251565e-16
Full-grid stencil passed: True


In [65]:

import json
import os

with open("submission_manifest.json") as f:
    manifest = json.load(f)

print("=== MANIFEST ===")
print(json.dumps(manifest, indent=2))

print("\n=== FILE CHECK ===")
for name in [
    "distributed_cuda_solver.py",
    "submission_manifest.json"
]:
    print(name, "EXISTS" if os.path.exists(name) else "MISSING")


=== MANIFEST ===
{
  "github_username": "AidynTt",
  "gpu_uuid": "GPU-5f3e9715-89c0-2858-ddd4-2c94ecde93ed",
  "instance_seed": 15483842694073859920,
  "mpi_ranks": 4,
  "cartesian_grid": [
    2,
    2
  ],
  "global_grid": [
    512,
    512
  ],
  "time_steps": 400,
  "serialized_ms": 8065.382182000121,
  "overlapped_ms": 2235.6187339996723,
  "speedup": 3.6076733744177436,
  "parity_max_diff": 0.0,
  "parity_passed": true,
  "ptx_counts_rank0": {
    "st.shared": 6,
    "ld.shared": 5,
    "bar.sync": 1
  },
  "ptx_audit_passed": true,
  "serialized_trace": [
    {
      "step": 0,
      "energy": 0.03141592560659228,
      "residual": 0.0
    },
    {
      "step": 100,
      "energy": 0.031416674487576915,
      "residual": 1.1209713856619017e-05
    },
    {
      "step": 200,
      "energy": 0.03141742952835748,
      "residual": 1.0986181973573096e-05
    },
    {
      "step": 300,
      "energy": 0.0314181904224978,
      "residual": 1.077027204579787e-05
    },
    {
      

# Analytical Questions Q1–Q6

### Q1. Memory Coalescing and Column Packing

In our program we use 258 × 258 array. Each number takes 8 bytes because we use float64.

The left and right columns are not stored together in GPU memory. There is 2064 bytes between two elements in one column. Because of this we cannot copy the column like one normal block.

We use `pack_halos_kernel` to put these numbers in one buffer. One column needs only 2048 bytes.

If we copy the full rectangle, it will use 528384 bytes for one column. So we will transfer 526336 extra bytes. Packing is better because we send less data.

### Q2. Shared Memory and Bank Conflicts

We use 16 × 16 CUDA threads and 18 × 18 shared memory tile.

For thread `(0,0)`, the center is `smem[1,1]`. It reads other values from `smem[0,1]`, `smem[2,1]`, `smem[1,0]` and `smem[1,2]`.

For thread `(15,15)`, center is `smem[16,16]`. It reads values from `smem[15,16]`, `smem[17,16]`, `smem[16,15]` and `smem[16,17]`.

Tesla T4 has 32 memory banks. Each bank has 4 bytes width. Our float64 number needs 8 bytes.

When threads read values in one row, memory access is usually better. For column access, different threads can use the same banks. This can make bank conflicts and slow down the program.

### Q3. PTX and Synchronization

I checked PTX instructions from my CUDA kernel.

My results:

- `st.shared` = 6
- `ld.shared` = 5
- `bar.sync` = 1

The PTX test passed.

We have 256 threads in one block, so it is 8 warps.

We need `cuda.syncthreads()` because threads must finish writing data to shared memory before other threads read it.

Without this synchronization, some threads can read old or wrong values. Then our calculation can give wrong result.

### Q4. GPU Sharing and Performance

In this lab I used 4 MPI processes on one Tesla T4 GPU.

All processes share the same GPU. So sometimes they need to wait for GPU resources.

I tested two modes.

Serialized time was **8065.382 ms**.

Overlapped time was **2235.619 ms**.

Speedup was **3.61 times**.

Overlapped mode was much faster in my test. It can calculate inside cells while MPI communication is working.

But all 4 processes still use one GPU. If we have 4 different GPUs with InfiniBand, they can do calculations on separate GPUs and performance can be different.

### Q5. MPI Deadlock and PROC_NULL

If we use blocking `Send` and then `Recv`, we can have deadlock.

For small messages MPI can save data in buffer. But for big messages, `Send` can wait for another process to start receiving.

If all processes are waiting to send, nobody starts receiving. Then program can stop and wait forever.

In my code I used `Isend` and `Irecv`. After this I used `Waitall`. It helps to avoid this problem.

`MPI.PROC_NULL` means there is no neighbor in this direction.

For example, rank `(0,0)` has no upper and left neighbor. We do not need to send data there. We just use zero for outside boundary.

### Q6. CFL Stability and Energy

Our grid size is 512 × 512.

The distance between points is:

\[
dx=dy=1/513\approx0.0019493
\]

Maximum diffusivity is about 0.24.

The maximum stable time step for diffusion is:

\[
dt_{max}=\frac{dx^2}{4(0.24)}
\approx0.0000039584
\]

In my program I used:

\[
dt\approx0.0000015834
\]

My time step is smaller than maximum value. So it follows the diffusion CFL condition.

I also checked energy every 100 steps.

| Step | Energy |
|---|---|
| 0 | 0.0314159256 |
| 100 | 0.0314166745 |
| 200 | 0.0314174295 |
| 300 | 0.0314181904 |
| 400 | 0.0314189569 |

Energy increased a little bit. I think this is because our equation has a heat source which adds energy.

Also residual became smaller. At step 100 it was about 0.00001121 and at step 400 it was about 0.00001056.

This shows the calculation was stable during our 400 steps, but it does not prove stability for all cases.

At the end I compared serialized and overlapped results.

The maximum difference was **0.0**, which is smaller than `1e-12`.

So both modes gave the same result in my test.
